<h1 style="color:#2E86C1; font-weight:bold; text-align:center; margin-top:20px; margin-bottom:30px;">02 - Feature Engineering</h1>

<h2 style="color:#2874A6; font-weight:bold;">1. Introduction</h2>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In this notebook, feature engineering is performed on the CMOS sensor dataset in order to transform raw 32×32 sensor frames into a more informative and compact representation for downstream analysis and modelling.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Based on the findings from the data understanding phase, the dataset exhibits strong temporal characteristics, suggesting that meaningful patterns evolve over time. Additionally, the data contains artefacts such as zero-valued frames and occasional saturation, which should be taken into account during feature construction. Therefore, the feature engineering process focuses primarily on capturing temporal signal dynamics rather than relying solely on raw spatial pixel values.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Instead of directly using all 1024 pixel values per frame, higher-level statistical and time-aware features are extracted. This approach reduces dimensionality, improves interpretability, and aims to retain the most relevant aspects of the sensor response while supporting downstream machine learning tasks.
</p>

<h2 style="color:#2874A6; font-weight:bold;">2. Feature Engineering Objective</h2>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The goal of this phase is to extract meaningful and informative features from the sensor data that aim to represent the behaviour of the signal over time.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Since the dataset consists of temporally ordered sensor frames, the feature engineering strategy focuses primarily on capturing temporal dynamics rather than relying solely on raw spatial pixel values.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In particular, the extracted features are designed to approximate different stages of the sensor signal, such as baseline behaviour, response intensity, peak characteristics, and recovery patterns. These stages are treated as heuristic representations rather than exact ground truth phases.
</p>

<ul style="font-size:17px; line-height:1.8; max-width:900px; margin:auto;">
  <li>Baseline signal characteristics</li>
  <li>Response magnitude and peak behaviour</li>
  <li>Temporal dynamics such as rate of change</li>
  <li>Recovery behaviour and signal stabilization</li>
  <li>Noise and signal variability</li>
</ul>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
This structured representation reduces dimensionality and improves interpretability, making the data more suitable for downstream preprocessing and machine learning tasks.
</p>

<h2 style="color:#2874A6; font-weight:bold;">3. Data Preparation for Feature Extraction</h2>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Before extracting features, the sensor data is organised into a structure that supports temporal analysis. Each observation corresponds to a frame captured by a 32×32 CMOS sensor, where the spatial grid is flattened into 1024 pixel values.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The dataset is first loaded and inspected to verify its shape, column structure, and overall consistency. Special attention is given to identifying pixel-related columns and separating them from any metadata or auxiliary variables, ensuring that feature extraction is applied only to relevant signal data.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Since the data represents a sequence of temporally ordered frames, preserving the original ordering is essential for capturing signal dynamics. This preparation step ensures that the dataset is clean, well-structured, and suitable for constructing time-aware features in subsequent stages.
</p>

In [3]:
# ===============================
# Core Libraries
# ===============================
import pandas as pd
import numpy as np
from pathlib import Path

# ===============================
# Visualization
# ===============================
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

# ===============================
# Statistics & Analysis
# ===============================

# ===============================
# Optional: Time Handling
# ===============================

# ===============================
# Warnings (clean notebook output)
# ===============================
import warnings
warnings.filterwarnings("ignore")

# ===============================
# Display Settings (for Jupyter)
# ===============================
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 20)

# ===============================
# Quick Check
# ===============================
print("Libraries loaded successfully ✅")

Libraries loaded successfully ✅


In [4]:
# ===============================
# Define Dataset Path (Relative)
# ===============================
DATA_PATH = Path("../../Tolouene/dataset")

# ===============================
# Check Path Availability
# ===============================
if DATA_PATH.exists():
    print("Dataset path exists:", DATA_PATH.resolve())

    print("\nContents:")
    for item in DATA_PATH.iterdir():
        print("-", item.name)
else:
    print("Dataset path not found. Please check the relative path.")

Dataset path exists: E:\Saxion\Projects\DataOps\Group\group-4\Tolouene\dataset

Contents:
- tolouene_aug_2024.dat


In [5]:
# Search for .dat files in dataset (including subfolders)
dat_files = list(DATA_PATH.rglob("*.dat"))

print("Number of .dat files:", len(dat_files))

# Show files
for f in dat_files:
    print("-", f)

# Select the first file
sample_file = dat_files[0] if dat_files else None

print("\nSample file selected:", sample_file)

Number of .dat files: 1
- ..\..\Tolouene\dataset\tolouene_aug_2024.dat

Sample file selected: ..\..\Tolouene\dataset\tolouene_aug_2024.dat


In [6]:
sample_file = None

for ext in ["*.csv", "*.tsv", "*.txt"]:
    files = list(DATA_PATH.glob(ext))
    if files:
        sample_file = files[0]
        break

if sample_file is not None:
    print("Sample file selected:", sample_file.name)
else:
    print("No suitable data file found in the dataset directory.")

No suitable data file found in the dataset directory.


In [7]:
if sample_file is not None:
    with open(sample_file, "r", encoding="utf-8", errors="ignore") as f:
        for i in range(10):
            line = f.readline()
            if not line:
                break
            print(f"Line {i+1}:", repr(line))
else:
    print("No file available to preview.")

No file available to preview.


In [29]:
rows = []

with open(sample_file, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()[2:]

for line in lines:
    values = line.strip().split()
    if values:
        rows.append(values)

df = pd.DataFrame(rows)
df = df.apply(pd.to_numeric, errors="coerce")

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,...,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,1695.0,1686.0,1697.0,1692.0,1707.0,1684.0,1684.0,1700.0,1737.0,1714.0,1710.0,1708.0,1708.0,1717.0,1714.0,1721.0,1700.0,1682.0,1690.0,1670.0,1678.0,1672.0,1682.0,1696.0,1694.0,...,1703.0,1706.0,1713.0,1708.0,1699.0,1694.0,1697.0,1699.0,1701.0,1700.0,1708.0,1708.0,1695.0,1702.0,1704.0,1698.0,1690.0,1701.0,1679.0,1696.0,1711.0,1695.0,1696.0,1687.0,0.0
1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,...,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,4095.0,0.0


In [12]:
sample_file = None

for ext in ["*.csv", "*.CSV", "*.tsv", "*.txt", "*.dat"]:
    files = sorted(DATA_PATH.glob(ext))
    if files:
        sample_file = files[0]
        break

if sample_file is not None:
    print("Sample file selected:", sample_file.name)
else:
    print("No suitable data file found in the dataset directory.")

Sample file selected: tolouene_aug_2024.dat


<h3 style="color:#2E86C1; font-weight:bold;">3.1 Parsing the Raw Sensor File</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The raw sensor data is stored in a custom <code>.dat</code> format rather than a standard tabular structure. Each observation consists of multiple lines, including a timestamp, metadata, and sensor readings. To make the data suitable for analysis, the file is parsed and transformed into a structured dataframe.
</p>

In [23]:
# ===============================
# Parse the raw .dat sensor file
# Keep only the primary sensor frame (frame_1)
# ===============================
if sample_file is not None:
    with open(sample_file, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]

    print(f"Total non-empty lines: {len(lines)}")

    pixel_rows = []
    timestamps = []
    metadata_rows = []

    invalid_blocks = 0
    incomplete_blocks = 0

    for i in range(0, len(lines), 7):
        block = lines[i:i+7]

        if len(block) < 7:
            incomplete_blocks += 1
            continue

        timestamp_line = block[0]
        metadata_line = block[1]
        frame_1_line = block[2]

        frame_1_values = frame_1_line.split("\t")

        if len(frame_1_values) != 1024:
            invalid_blocks += 1
            continue

        frame_1_values = pd.to_numeric(frame_1_values, errors="coerce")

        pixel_rows.append(frame_1_values)
        timestamps.append(timestamp_line)
        metadata_rows.append(metadata_line)

    pixel_df = pd.DataFrame(pixel_rows, columns=[f"pixel_{i}" for i in range(1024)])

    df = pd.DataFrame({
        "timestamp": timestamps,
        "metadata": metadata_rows
    })

    df = pd.concat([df, pixel_df], axis=1)

    print("\nDataset loaded successfully.")
    print(f"Valid frames parsed: {len(df)}")
    print(f"Invalid blocks skipped: {invalid_blocks}")
    print(f"Incomplete blocks skipped: {incomplete_blocks}")
    print(f"Dataset shape: {df.shape}")

Total non-empty lines: 56875

Dataset loaded successfully.
Valid frames parsed: 8125
Invalid blocks skipped: 0
Incomplete blocks skipped: 0
Dataset shape: (8125, 1026)


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The parsing results indicate that the raw sensor file has been successfully transformed into a structured dataset. A total of 8125 valid observations were extracted, with no invalid or incomplete blocks detected. This suggests that the applied parsing logic is consistent with the underlying structure of the raw data.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Each observation consists of 1026 columns, including a timestamp, metadata information, and 1024 pixel values representing the flattened 32×32 primary sensor frame. This confirms that the dimensionality of the sensor data has been preserved during the parsing process.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The absence of invalid or incomplete records indicates that the dataset is structurally consistent and suitable for further preprocessing and feature engineering. In particular, the availability of a large number of observations supports meaningful temporal analysis, which aligns with earlier findings that temporal patterns are more informative than spatial variations in this dataset.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">3.2 Dataset Validation and Preparation</h3>

In [24]:
# Quick preview of the parsed dataset
df.head()

,timestamp,metadata,pixel_0,pixel_1,pixel_2,pixel_3,pixel_4,pixel_5,pixel_6,pixel_7,pixel_8,pixel_9,pixel_10,pixel_11,pixel_12,pixel_13,pixel_14,pixel_15,pixel_16,pixel_17,pixel_18,pixel_19,pixel_20,pixel_21,pixel_22,...,pixel_999,pixel_1000,pixel_1001,pixel_1002,pixel_1003,pixel_1004,pixel_1005,pixel_1006,pixel_1007,pixel_1008,pixel_1009,pixel_1010,pixel_1011,pixel_1012,pixel_1013,pixel_1014,pixel_1015,pixel_1016,pixel_1017,pixel_1018,pixel_1019,pixel_1020,pixel_1021,pixel_1022,pixel_1023
0,13-Aug-2024 14:08:38.158,33\t1,1695,1686,1697,1692,1707,1684,1684,1700,1737,1714,1710,1708,1708,1717,1714,1721,1700,1682,1690,1670,1678,1672,1682,...,1703,1706,1713,1708,1699,1694,1697,1699,1701,1700,1708,1708,1695,1702,1704,1698,1690,1701,1679,1696,1711,1695,1696,1687,0
1,13-Aug-2024 14:08:39.806,34\t1,1696,1684,1698,1692,1708,1685,1685,1700,1737,1713,1708,1707,1707,1716,1713,1723,1699,1681,1690,1669,1676,1671,1683,...,1703,1706,1713,1707,1698,1693,1698,1698,1700,1698,1709,1706,1694,1702,1703,1697,1689,1701,1677,1695,1710,1696,1695,1686,0
2,13-Aug-2024 14:08:42.666,35\t1,1695,1684,1696,1692,1706,1683,1683,1699,1737,1713,1708,1707,1707,1717,1713,1721,1698,1682,1689,1670,1677,1672,1682,...,1702,1706,1713,1708,1698,1694,1697,1699,1700,1697,1708,1704,1695,1702,1704,1697,1690,1701,1677,1695,1708,1696,1696,1688,0
3,13-Aug-2024 14:08:44.160,36\t1,1696,1684,1697,1692,1706,1683,1683,1699,1736,1713,1708,1706,1706,1716,1713,1721,1698,1683,1689,1669,1677,1671,1684,...,1702,1706,1713,1707,1699,1692,1699,1699,1701,1698,1709,1706,1695,1701,1705,1697,1690,1700,1678,1695,1709,1696,1696,1687,0
4,13-Aug-2024 14:08:45.507,37\t1,1695,1686,1697,1692,1707,1683,1684,1700,1737,1713,1709,1706,1707,1716,1712,1722,1699,1682,1691,1668,1677,1671,1681,...,1703,1708,1713,1709,1701,1696,1699,1701,1702,1700,1710,1707,1694,1703,1705,1697,1690,1701,1678,1696,1711,1697,1697,1688,0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The initial inspection of the parsed dataset confirms that the data has been successfully transformed into a structured tabular format. Each observation consists of a timestamp, metadata, and 1024 pixel values representing the flattened sensor frame. This structure aligns with the expected format derived from the raw <code>.dat</code> file.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
To ensure that feature extraction is applied only to relevant signal data, the pixel columns are explicitly separated from metadata and timestamp information. The resulting pixel-only dataframe contains 1024 columns, confirming that the spatial resolution of the original 32×32 sensor grid has been preserved.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In addition, the timestamp column is converted into a datetime format to support temporal analysis. No missing values were introduced during this conversion, indicating that the timestamp data is consistent and well-formed.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, the dataset is now clean, well-structured, and ready for feature engineering. In particular, isolating the pixel data enables the extraction of meaningful statistical and temporal features without interference from non-numeric attributes.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">3.3 Data Structure and Data Types</h3>

In [27]:
# Check dataset structure and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8125 entries, 0 to 8124
Columns: 1026 entries, timestamp to pixel_1023
dtypes: int64(1024), object(2)
memory usage: 63.6+ MB


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The dataset contains 8125 observations, each representing a single sensor frame captured over time. The dataframe includes a total of 1026 columns, consisting of two non-numeric columns (timestamp and metadata) and 1024 numeric pixel features corresponding to the flattened 32×32 sensor grid.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The pixel values are stored as integer data types, which indicates that the sensor measurements have been successfully parsed and converted into a numerical format suitable for further analysis. In contrast, the timestamp and metadata columns are stored as object types, reflecting their non-numeric nature.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The overall memory usage of approximately 63.6 MB suggests that the dataset is moderately large, which further justifies the need for dimensionality reduction through feature engineering. Instead of working directly with all 1024 pixel features, it is beneficial to extract compact statistical and temporal features that better capture the underlying signal behaviour.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">3.4 Pixel Data Isolation</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The results confirm that the pixel columns have been successfully isolated from the rest of the dataset. A total of 1024 pixel features were identified, which corresponds exactly to the flattened 32×32 sensor grid for each observation.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The resulting pixel-only dataframe has a shape of (8125, 1024), indicating that all observations have retained their full spatial resolution. This confirms that no pixel information has been lost during the parsing and preparation stages.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
By separating the pixel data from timestamp and metadata fields, the dataset is now properly structured for feature engineering. All subsequent statistical and temporal features can be computed exclusively on the sensor signal, ensuring accurate and meaningful representation of the underlying data.
</p>

In [28]:
# ===============================
# Select pixel columns only
# ===============================
pixel_cols = [col for col in df.columns if col.startswith("pixel_")]

if len(pixel_cols) == 1024:
    df_pixels = df[pixel_cols].copy()
    print("Pixel columns successfully selected.")
else:
    print("Warning: Unexpected number of pixel columns:", len(pixel_cols))

print("Number of pixel columns:", len(pixel_cols))
print("Pixel-only dataframe shape:", df_pixels.shape)

Pixel columns successfully selected.
Number of pixel columns: 1024
Pixel-only dataframe shape: (8125, 1024)


<h3 style="color:#2E86C1; font-weight:bold;">3.5 Timestamp Validation</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The results show that no missing values were introduced during the timestamp conversion process, as indicated by zero missing timestamps. This confirms that all timestamp entries in the dataset are valid and correctly formatted.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The absence of invalid or missing timestamps suggests that the temporal information in the dataset is consistent and reliable. This is particularly important for subsequent analysis, as it ensures that temporal patterns and trends can be accurately captured without the need for additional data cleaning.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, the successful conversion and validation of the timestamp column further confirms that the dataset is well-prepared for time-aware feature engineering and analysis.
</p>

In [29]:
# Convert timestamp column to datetime
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%d-%b-%Y %H:%M:%S.%f",
    errors="coerce"
)

print("Missing timestamps after conversion:", df["timestamp"].isna().sum())

Missing timestamps after conversion: 0


<h3 style="color:#2E86C1; font-weight:bold;">3.6 Missing Value Analysis Results</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The analysis shows that there are no missing values in the pixel data, as indicated by a total count of zero missing entries. This confirms that all sensor measurements are complete and have been successfully preserved during the data parsing and preparation process.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The absence of missing values ensures that no additional imputation or data cleaning is required before feature engineering. This is particularly important for statistical feature extraction, as it allows all computations to be performed directly on the full dataset without introducing bias or uncertainty.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, the completeness of the pixel data further confirms that the dataset is well-prepared for subsequent analysis, enabling reliable extraction of meaningful temporal and statistical features.
</p>

In [30]:
# Check missing values in pixel data
missing_pixels = df_pixels.isna().sum().sum()
print("Total missing pixel values:", missing_pixels)

Total missing pixel values: 0


<h3 style="color:#2E86C1; font-weight:bold;">3.7 Summary of Data Preparation</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
At this stage, the raw sensor data has been successfully parsed and transformed into a structured dataframe. The primary sensor frame was retained, the pixel columns were isolated, the timestamp was converted into a datetime format, and the dataset was checked for missing values.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
These preparation steps confirm that the dataset is clean, consistent, and suitable for feature extraction. The next phase therefore focuses on constructing compact and meaningful features from the pixel-level sensor data.
</p>

<h2 style="color:#2874A6; font-weight:bold;">4. Feature Engineering</h2>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Following the data preparation phase, where the raw sensor data was parsed, validated, and structured, the dataset is now ready for feature engineering. At this stage, the focus shifts from understanding and cleaning the data to transforming it into a form that better supports analysis and modelling.
</p>
<h3 style="color:#2E86C1; font-weight:bold;">4.1 Introduction</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In this phase, feature engineering is applied to transform the prepared sensor data into a more compact and informative representation. While the raw dataset contains 1024 pixel values per observation, directly using all pixel features may not be efficient for analysis and modelling.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Based on the data understanding phase, the dataset exhibits strong temporal characteristics, suggesting that patterns in the signal evolve over time. Therefore, the feature engineering process focuses on extracting statistical and time-aware features that capture the dynamics of the sensor signal rather than relying solely on spatial pixel values.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The goal is to reduce dimensionality while retaining key characteristics of the signal, enabling more effective downstream analysis and machine learning. These engineered features provide a simplified yet meaningful representation of the underlying sensor behaviour.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">4.2 Frame-Level Feature Extraction</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
To reduce the high dimensionality of the dataset, each sensor frame is summarised using statistical features computed from the pixel values. Instead of directly using all 1024 pixel values, these features provide a compact representation of the signal while preserving key characteristics such as intensity, variability, and range.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
These statistical features serve as the first step in transforming the raw sensor data into a more interpretable and computationally efficient format for further analysis.
</p>

In [34]:
# ===============================
# Frame-level statistical features
# ===============================
df_features = pd.DataFrame(index=df.index)

df_features["frame_mean"] = df_pixels.mean(axis=1)
df_features["frame_std"] = df_pixels.std(axis=1)
df_features["frame_min"] = df_pixels.min(axis=1)
df_features["frame_max"] = df_pixels.max(axis=1)
df_features["frame_median"] = df_pixels.median(axis=1)

df_features.head()

,frame_mean,frame_std,frame_min,frame_max,frame_median
0,1695.944336,56.049097,0,1760,1695.0
1,1695.953125,56.021290,0,1759,1695.0
2,1695.674805,56.026585,0,1759,1695.0
3,1696.091797,56.074498,0,1759,1695.0
4,1696.105469,56.046750,0,1760,1695.0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The extracted frame-level features show that the sensor signal remains relatively stable across consecutive observations. The mean values are consistently around 1695–1696, indicating that the overall signal intensity does not fluctuate significantly within this segment of the data.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The standard deviation values are also highly consistent, remaining close to 56 for all frames. This suggests that the variability within each frame is stable, and the distribution of pixel values does not change substantially over time.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The minimum values are consistently zero, which may reflect the presence of low-intensity or inactive pixels within the sensor grid. In contrast, the maximum values are around 1759–1760, indicating the upper range of the signal intensity in this portion of the dataset.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, these results suggest that the sensor is operating under relatively steady conditions, likely corresponding to a baseline or stable phase. The statistical features effectively summarise this behaviour while significantly reducing the dimensionality of the data.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">4.3 Temporal Trend Features</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
While frame-level statistical features summarise each observation individually, they do not directly capture how the signal changes over time. To better represent temporal dynamics, additional trend-based features are extracted from consecutive observations.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
These features quantify short-term changes in signal intensity and variability, making them particularly useful for identifying transitions, response behaviour, and recovery patterns in the sensor data.
</p>

In [36]:
# ===============================
# Temporal Trend Features
# ===============================

# Compute first-order differences (change between consecutive frames)
df_features["diff_mean"] = df_features["frame_mean"].diff()
df_features["diff_std"] = df_features["frame_std"].diff()
df_features["diff_min"] = df_features["frame_min"].diff()
df_features["diff_max"] = df_features["frame_max"].diff()

# Handle NaN values introduced by diff (first row)
df_features[["diff_mean", "diff_std", "diff_min", "diff_max"]] = (
    df_features[["diff_mean", "diff_std", "diff_min", "diff_max"]]
    .fillna(0)
)

# Preview results
df_features.head()

,frame_mean,frame_std,frame_min,frame_max,frame_median,diff_mean,diff_std,diff_max,diff_min
0,1695.944336,56.049097,0,1760,1695.0,0.000000,0.000000,0.0,0.0
1,1695.953125,56.021290,0,1759,1695.0,0.008789,-0.027808,-1.0,0.0
2,1695.674805,56.026585,0,1759,1695.0,-0.278320,0.005295,0.0,0.0
3,1696.091797,56.074498,0,1759,1695.0,0.416992,0.047913,0.0,0.0
4,1696.105469,56.046750,0,1760,1695.0,0.013672,-0.027749,1.0,0.0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The temporal trend features show that changes in the sensor signal between consecutive observations are relatively small. The differences in the mean values are close to zero, indicating that the overall signal intensity evolves gradually rather than exhibiting abrupt shifts.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Similarly, the changes in standard deviation are minimal, suggesting that the variability within each frame remains stable over time. This reinforces the observation that the distribution of pixel values does not fluctuate significantly between consecutive frames.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The changes in maximum values are limited to small increments (e.g., ±1), while the minimum values remain constant at zero, resulting in no variation in the lower bound of the signal. These patterns further indicate that the sensor is operating in a steady state with only minor fluctuations.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, these results suggest that the signal transitions smoothly over time, without sudden spikes or drops. The temporal features therefore provide a useful representation of signal dynamics, capturing subtle changes that are not visible in static frame-level summaries alone.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">4.4 Handling Invalid Frames</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
During the feature extraction process, several patterns may indicate that certain frames do not reflect meaningful sensor behaviour. These include frames with extremely low variability, constant signals, or values that may suggest sensor saturation.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Such frames can reduce the quality of the dataset and negatively affect downstream analysis and modelling. Therefore, it is useful to identify and filter out potentially invalid or non-informative frames before proceeding further.
</p>

In [37]:
# ===============================
# Identify and remove invalid frames
# ===============================

# Define threshold for low variability
std_threshold = 1  # small value → nearly constant signal

# Create mask for valid frames
valid_mask = df_features["frame_std"] > std_threshold

# Filter dataset
df_features_clean = df_features[valid_mask].copy()
df_clean = df[valid_mask].copy()

# Report results
print("Original number of frames:", len(df_features))
print("Remaining valid frames:", len(df_features_clean))
print("Removed frames:", len(df_features) - len(df_features_clean))

Original number of frames: 8125
Remaining valid frames: 8125
Removed frames: 0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The filtering results show that no frames were removed from the dataset, as all observations satisfied the variability threshold. This indicates that none of the frames exhibit extremely low variability or constant signal behaviour based on the applied criterion.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
This suggests that the primary sensor frames retained during the parsing stage already represent meaningful and informative signal patterns. In particular, the earlier decision to exclude auxiliary frames (such as zero-valued or saturated frames) appears to have effectively removed most non-informative data.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
As a result, no additional filtering is required at this stage, and the dataset can be used directly for further feature analysis. This outcome supports the consistency and quality of the prepared data.
</p>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
This result suggests that the dataset is already well-structured and does not require additional filtering at this stage.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">4.5 Rolling Features</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In addition to frame-level and difference-based features, rolling statistics are used to capture short-term local trends in the sensor signal. Unlike single-frame summaries, rolling features smooth small fluctuations and provide a more stable representation of temporal behaviour.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
These features are particularly useful for identifying gradual changes in signal intensity and local variability over consecutive observations. As a result, they provide complementary information to both static and difference-based features.
</p>

In [38]:
# ===============================
# Rolling Features
# ===============================

window_size = 5

df_features_clean["rolling_mean"] = (
    df_features_clean["frame_mean"].rolling(window=window_size, min_periods=1).mean()
)

df_features_clean["rolling_std"] = (
    df_features_clean["frame_mean"].rolling(window=window_size, min_periods=1).std().fillna(0)
)

df_features_clean["rolling_max"] = (
    df_features_clean["frame_max"].rolling(window=window_size, min_periods=1).max()
)

df_features_clean["rolling_min"] = (
    df_features_clean["frame_min"].rolling(window=window_size, min_periods=1).min()
)

df_features_clean.head()

,frame_mean,frame_std,frame_min,frame_max,frame_median,diff_mean,diff_std,diff_max,diff_min,rolling_mean,rolling_std,rolling_max,rolling_min
0,1695.944336,56.049097,0,1760,1695.0,0.000000,0.000000,0.0,0.0,1695.944336,0.000000,1760.0,0.0
1,1695.953125,56.021290,0,1759,1695.0,0.008789,-0.027808,-1.0,0.0,1695.948730,0.006215,1760.0,0.0
2,1695.674805,56.026585,0,1759,1695.0,-0.278320,0.005295,0.0,0.0,1695.857422,0.158212,1760.0,0.0
3,1696.091797,56.074498,0,1759,1695.0,0.416992,0.047913,0.0,0.0,1695.916016,0.174414,1760.0,0.0
4,1696.105469,56.046750,0,1760,1695.0,0.013672,-0.027749,1.0,0.0,1695.953906,0.173187,1760.0,0.0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The rolling features provide a smoothed representation of the sensor signal over short time windows. Compared to the frame-level features, the rolling mean values vary more gradually, indicating that short-term fluctuations in the signal are effectively smoothed.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The rolling standard deviation values remain relatively low, suggesting that local variability within the signal is limited and stable over time. This further supports the observation that the signal does not exhibit abrupt changes within small temporal windows.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The rolling maximum and minimum values remain largely consistent, with the minimum staying at zero and the maximum fluctuating only slightly. This indicates that the overall range of the signal remains stable within the selected window, reinforcing the presence of steady operating conditions.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, these rolling features provide a more robust and stable representation of the sensor behaviour by capturing local temporal trends while reducing the impact of noise and minor fluctuations. This makes them particularly useful for downstream analysis and modelling.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">4.6 Phase-Aware Feature Exploration (Optional)</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In addition to general temporal features, a simple phase-based analysis can be explored to approximate different stages of the sensor signal, such as baseline, response, and recovery. It is important to note that these phases are heuristic approximations rather than ground truth labels.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The segmentation is based on variations in signal intensity over time, allowing a coarse interpretation of how the sensor behaves across different operating conditions. This approach provides additional insight into the temporal structure of the data without making strong assumptions about exact phase boundaries.
</p>


In [40]:
# ===============================
# Phase-Aware Feature Exploration
# ===============================

# Define heuristic thresholds based on frame-level mean intensity
low_threshold = df_features_clean["frame_mean"].quantile(0.33)
high_threshold = df_features_clean["frame_mean"].quantile(0.66)

# Assign approximate phase labels
def assign_phase(value):
    if value <= low_threshold:
        return "baseline"
    elif value <= high_threshold:
        return "transition"
    else:
        return "response"

df_features_clean["phase"] = df_features_clean["frame_mean"].apply(assign_phase)

# Preview the resulting phase labels
df_features_clean[["frame_mean", "phase"]].head()

# Simple summary of phase distribution
phase_summary = df_features_clean["phase"].value_counts()
print(phase_summary)

# Mean signal intensity per approximate phase
phase_mean_summary = df_features_clean.groupby("phase")["frame_mean"].mean()
print(phase_mean_summary)

phase
response      2760
transition    2683
baseline      2682
Name: count, dtype: int64
phase
baseline      1695.631880
response      1697.370919
transition    1696.527158
Name: frame_mean, dtype: float64


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The phase distribution shows that the observations are relatively evenly divided across the three heuristic categories, with approximately similar counts for baseline, transition, and response phases. This indicates that the chosen quantile-based thresholds produce a balanced segmentation of the signal.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The average signal intensity differs slightly across the phases. The baseline phase exhibits the lowest mean value (approximately 1695.63), while the response phase shows the highest mean value (approximately 1697.37), with the transition phase lying in between. This ordering is consistent with the intended interpretation of the phases based on signal intensity.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
However, the differences between phase means are relatively small, suggesting that the signal variations are subtle rather than drastic. This observation aligns with earlier findings that the sensor signal evolves smoothly over time without sharp transitions.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, the phase-based segmentation provides a coarse but meaningful grouping of observations based on signal intensity. It should be interpreted as an approximate categorisation of signal regimes rather than an exact representation of physical sensor states.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">4.7 Feature Summary</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In this phase, a comprehensive set of features was extracted to represent the sensor signal in a more compact and informative form. These include frame-level statistical features, temporal difference-based features, rolling statistics, and a simple phase-aware segmentation.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The frame-level features capture the overall characteristics of each observation, while the temporal features describe how the signal evolves between consecutive frames. In addition, the rolling features provide a smoothed representation of short-term trends, reducing the impact of noise and minor fluctuations.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The phase-aware segmentation offers an additional perspective by grouping observations into approximate signal regimes based on intensity levels. It is important to note that these phase labels are heuristic approximations rather than ground truth categories.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Furthermore, an evaluation of potential invalid frames indicated that no additional filtering was required, suggesting that the dataset is already well-structured and contains meaningful signal patterns.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, these features provide a multi-level representation of the sensor data, capturing both static properties and temporal dynamics. This structured feature set forms a strong foundation for subsequent preprocessing and modelling steps.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The phase-based features provide a detailed view of how the sensor signal evolves across different stages of operation. The baseline mean (approximately 2138) represents the normal operating condition of the sensor before exposure, although the relatively high standard deviation suggests that some variability is present even in this phase.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
During the exposure phase, the mean signal increases to around 2612, indicating a clear response of the sensor to the external stimulus. The reduction in standard deviation compared to the baseline phase suggests a more stable but elevated signal level during exposure.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In the recovery phase, the mean signal further increases to approximately 3086 instead of returning to baseline levels. This unexpected behaviour indicates that the signal may not have fully stabilized, or that the segmentation of phases may not perfectly align with the true physical process.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The response amplitude (~474) confirms that the sensor exhibits a noticeable increase in signal during exposure compared to baseline. However, the negative recovery drop indicates that the signal continues to increase rather than decrease after exposure, which further suggests that the recovery phase is not properly captured or that the signal segment includes part of the exposure period.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Additionally, the large positive final offset (~1953) shows that the final signal value remains significantly higher than the baseline, reinforcing the observation that the sensor does not return to its initial state within the current segment.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, these results highlight that while phase-based feature extraction provides meaningful insights, the current segmentation strategy based on fixed proportions may not accurately reflect the true boundaries of baseline, exposure, and recovery phases. This suggests that a more adaptive or data-driven segmentation approach could further improve the quality of the extracted features.
</p>

<h2 style="color:#2874A6; font-weight:bold;">5. Preprocessing</h2>

<h3 style="color:#2E86C1; font-weight:bold;">5.1 Introduction</h3>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Following the feature engineering phase, the dataset is further prepared for machine learning tasks. While the extracted features provide a meaningful representation of the sensor signal, additional preprocessing steps are required to ensure that the data is suitable for modelling.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In particular, the data must be transformed into a format where multiple observations are available for training and evaluation. This involves segmenting the signal into smaller windows and organising the data in a way that supports supervised learning.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
These preprocessing steps aim to improve the usability and structure of the dataset, providing a solid foundation for subsequent modelling and evaluation.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">5.2 Window-Based Feature Extraction</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Although the extracted features provide a meaningful representation of each frame, machine learning models typically require a structured set of samples that capture local temporal context. To achieve this, the feature sequence is divided into smaller overlapping windows.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Each window represents a short segment of the signal and is treated as an individual sample. By grouping consecutive observations, this approach preserves short-term temporal patterns while increasing the number of training instances available for later modelling.
</p>

In [42]:
# ===============================
# Window-Based Feature Extraction
# ===============================

# Select numeric features to include in each window
feature_columns = [
    "frame_mean", "frame_std", "frame_min", "frame_max", "frame_median",
    "diff_mean", "diff_std", "diff_max", "diff_min",
    "rolling_mean", "rolling_std", "rolling_max", "rolling_min"
]

window_size = 5
step_size = 1

X_windows = []

for start in range(0, len(df_features_clean) - window_size + 1, step_size):
    end = start + window_size
    window = df_features_clean[feature_columns].iloc[start:end].values.flatten()
    X_windows.append(window)

X_windows = np.array(X_windows)

print("Windowed feature matrix shape:", X_windows.shape)
window_feature_names = [
    f"{col}_t{i}"
    for i in range(window_size)
    for col in feature_columns
]

X_windowed_df = pd.DataFrame(X_windows, columns=window_feature_names)

X_windowed_df.head()

Windowed feature matrix shape: (8121, 65)


,frame_mean_t0,frame_std_t0,frame_min_t0,frame_max_t0,frame_median_t0,diff_mean_t0,diff_std_t0,diff_max_t0,diff_min_t0,rolling_mean_t0,rolling_std_t0,rolling_max_t0,rolling_min_t0,frame_mean_t1,frame_std_t1,frame_min_t1,frame_max_t1,frame_median_t1,diff_mean_t1,diff_std_t1,diff_max_t1,diff_min_t1,rolling_mean_t1,rolling_std_t1,rolling_max_t1,...,frame_std_t3,frame_min_t3,frame_max_t3,frame_median_t3,diff_mean_t3,diff_std_t3,diff_max_t3,diff_min_t3,rolling_mean_t3,rolling_std_t3,rolling_max_t3,rolling_min_t3,frame_mean_t4,frame_std_t4,frame_min_t4,frame_max_t4,frame_median_t4,diff_mean_t4,diff_std_t4,diff_max_t4,diff_min_t4,rolling_mean_t4,rolling_std_t4,rolling_max_t4,rolling_min_t4
0,1695.944336,56.049097,0.0,1760.0,1695.0,0.000000,0.000000,0.0,0.0,1695.944336,0.000000,1760.0,0.0,1695.953125,56.021290,0.0,1759.0,1695.0,0.008789,-0.027808,-1.0,0.0,1695.948730,0.006215,1760.0,...,56.074498,0.0,1759.0,1695.0,0.416992,0.047913,0.0,0.0,1695.916016,0.174414,1760.0,0.0,1696.105469,56.046750,0.0,1760.0,1695.0,0.013672,-0.027749,1.0,0.0,1695.953906,0.173187,1760.0,0.0
1,1695.953125,56.021290,0.0,1759.0,1695.0,0.008789,-0.027808,-1.0,0.0,1695.948730,0.006215,1760.0,0.0,1695.674805,56.026585,0.0,1759.0,1695.0,-0.278320,0.005295,0.0,0.0,1695.857422,0.158212,1760.0,...,56.046750,0.0,1760.0,1695.0,0.013672,-0.027749,1.0,0.0,1695.953906,0.173187,1760.0,0.0,1696.052734,56.052492,0.0,1760.0,1695.0,-0.052734,0.005743,0.0,0.0,1695.975586,0.178396,1760.0,0.0
2,1695.674805,56.026585,0.0,1759.0,1695.0,-0.278320,0.005295,0.0,0.0,1695.857422,0.158212,1760.0,0.0,1696.091797,56.074498,0.0,1759.0,1695.0,0.416992,0.047913,0.0,0.0,1695.916016,0.174414,1760.0,...,56.052492,0.0,1760.0,1695.0,-0.052734,0.005743,0.0,0.0,1695.975586,0.178396,1760.0,0.0,1695.953125,56.021918,0.0,1758.0,1695.0,-0.099609,-0.030574,-2.0,0.0,1695.975586,0.178396,1760.0,0.0
3,1696.091797,56.074498,0.0,1759.0,1695.0,0.416992,0.047913,0.0,0.0,1695.916016,0.174414,1760.0,0.0,1696.105469,56.046750,0.0,1760.0,1695.0,0.013672,-0.027749,1.0,0.0,1695.953906,0.173187,1760.0,...,56.021918,0.0,1758.0,1695.0,-0.099609,-0.030574,-2.0,0.0,1695.975586,0.178396,1760.0,0.0,1696.301758,56.058566,0.0,1760.0,1695.0,0.348633,0.036648,2.0,0.0,1696.100977,0.127088,1760.0,0.0
4,1696.105469,56.046750,0.0,1760.0,1695.0,0.013672,-0.027749,1.0,0.0,1695.953906,0.173187,1760.0,0.0,1696.052734,56.052492,0.0,1760.0,1695.0,-0.052734,0.005743,0.0,0.0,1695.975586,0.178396,1760.0,...,56.058566,0.0,1760.0,1695.0,0.348633,0.036648,2.0,0.0,1696.100977,0.127088,1760.0,0.0,1696.015625,56.026768,0.0,1759.0,1695.0,-0.286133,-0.031798,-1.0,0.0,1696.085742,0.132896,1760.0,0.0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The window-based transformation results in a feature matrix of shape (8121, 65), indicating that 8121 overlapping windows were generated from the original sequence. Each window represents a short segment of the signal and is treated as an individual sample.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Each sample consists of 65 features, which correspond to the concatenation of 13 engineered features across 5 consecutive time steps. This structure allows the model to capture short-term temporal dependencies within each window, rather than relying solely on single-frame information.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The overlapping window approach ensures that temporal continuity is preserved, as consecutive windows share most of their observations. This enables the dataset to retain fine-grained temporal patterns while significantly increasing the number of training samples.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, this transformation converts the dataset from a sequential representation into a structured, sample-based format suitable for machine learning. Each window captures both the statistical properties and the temporal evolution of the sensor signal, providing a rich input for downstream modelling.
</p>

In [43]:
# Save the multi-sample dataset for modelling
multi_df.to_csv("engineered_features_multi.csv", index=False)

print("Multi-sample feature dataset saved successfully.")

Multi-sample feature dataset saved successfully.


<h3 style="color:#2E86C1; font-weight:bold;">5.3 Feature Scaling</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
After window-based transformation, the dataset contains features with different numerical ranges. For example, some features represent average signal intensity, while others capture variability or short-term changes. These differences in scale can affect the behaviour of machine learning models.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
To improve comparability across features and support more stable model training, the windowed feature matrix is standardised using feature scaling. This ensures that each feature contributes more equally to the learning process.
</p>

In [43]:
# ===============================
# Feature Scaling
# ===============================

from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit and transform
X_scaled = scaler.fit_transform(X_windows)

# Convert to DataFrame for better readability
X_scaled_df = pd.DataFrame(X_scaled, columns=X_windowed_df.columns)

# Quick checks
print("Scaled feature matrix shape:", X_scaled.shape)
print("Mean (approx.):", np.round(X_scaled.mean(), 6))
print("Std (approx.):", np.round(X_scaled.std(), 6))

X_scaled_df.head()


Scaled feature matrix shape: (8121, 65)
Mean (approx.): 0.0
Std (approx.): 0.877058


,frame_mean_t0,frame_std_t0,frame_min_t0,frame_max_t0,frame_median_t0,diff_mean_t0,diff_std_t0,diff_max_t0,diff_min_t0,rolling_mean_t0,rolling_std_t0,rolling_max_t0,rolling_min_t0,frame_mean_t1,frame_std_t1,frame_min_t1,frame_max_t1,frame_median_t1,diff_mean_t1,diff_std_t1,diff_max_t1,diff_min_t1,rolling_mean_t1,rolling_std_t1,rolling_max_t1,...,frame_std_t3,frame_min_t3,frame_max_t3,frame_median_t3,diff_mean_t3,diff_std_t3,diff_max_t3,diff_min_t3,rolling_mean_t3,rolling_std_t3,rolling_max_t3,rolling_min_t3,frame_mean_t4,frame_std_t4,frame_min_t4,frame_max_t4,frame_median_t4,diff_mean_t4,diff_std_t4,diff_max_t4,diff_min_t4,rolling_mean_t4,rolling_std_t4,rolling_max_t4,rolling_min_t4
0,-0.727059,-0.886370,0.0,-0.355361,-0.136394,-0.000605,-0.000486,0.000000,0.0,-0.773405,-2.310041,-1.146269,0.0,-0.716071,-1.255577,0.0,-1.091663,-0.136682,0.021937,-1.324380,-0.820257,0.0,-0.767649,-2.259201,-1.146548,...,-0.549940,0.0,-1.092167,-0.137004,1.076612,2.280316,-0.000202,0.0,-0.812193,-0.856061,-1.147095,0.0,-0.523595,-0.918299,0.0,-0.356131,-0.137165,0.034830,-1.321525,0.819853,0.0,-0.761258,-0.866469,-1.147369,0.0
1,-0.715912,-1.255363,0.0,-1.091557,-0.136394,0.022105,-1.324347,-0.820191,0.0,-0.767471,-2.258189,-1.146269,0.0,-1.068964,-1.185310,0.0,-1.091663,-0.136682,-0.719834,0.251441,-0.000202,0.0,-0.890917,-0.990626,-1.146548,...,-0.918229,0.0,-0.355926,-0.137004,0.034690,-1.321526,0.819853,0.0,-0.761059,-0.866307,-1.147095,0.0,-0.590438,-0.842081,0.0,-0.356131,-0.137165,-0.136733,0.273256,-0.000202,0.0,-0.732006,-0.822977,-1.147369,0.0
2,-1.068912,-1.185099,0.0,-1.091557,-0.136394,-0.719749,0.251607,0.000000,0.0,-0.890766,-0.990030,-1.146269,0.0,-0.540244,-0.549503,0.0,-1.091663,-0.136682,1.076564,2.280231,-0.000202,0.0,-0.811815,-0.855403,-1.146548,...,-0.842012,0.0,-0.355926,-0.137004,-0.136862,0.272802,-0.000202,0.0,-0.731802,-0.822818,-1.147095,0.0,-0.716696,-1.247881,0.0,-1.828706,-0.137165,-0.257837,-1.456078,-1.640313,0.0,-0.732006,-0.822977,-1.147369,0.0
3,-0.540031,-0.549311,0.0,-1.091557,-0.136394,1.076850,2.280567,0.000000,0.0,-0.811646,-0.854852,-1.146269,0.0,-0.522909,-0.917726,0.0,-0.355518,-0.136682,0.034552,-1.321573,0.819853,0.0,-0.760662,-0.865646,-1.146548,...,-1.247804,0.0,-1.828407,-0.137004,-0.257957,-1.456041,-1.640313,0.0,-0.731802,-0.822818,-1.147095,0.0,-0.274791,-0.761465,0.0,-0.356131,-0.137165,0.900217,1.744914,1.639909,0.0,-0.562817,-1.251374,-1.147369,0.0
4,-0.522691,-0.917522,0.0,-0.355361,-0.136394,0.034722,-1.321540,0.820191,0.0,-0.760482,-0.865091,-1.146269,0.0,-0.589773,-0.841522,0.0,-0.355518,-0.136682,-0.137014,0.272739,-0.000202,0.0,-0.731394,-0.822173,-1.146548,...,-0.761398,0.0,-0.355926,-0.137004,0.900015,1.744043,1.639909,0.0,-0.562585,-1.251196,-1.147095,0.0,-0.637475,-1.183501,0.0,-1.092419,-0.137165,-0.739729,-1.514327,-0.820257,0.0,-0.583373,-1.202878,-1.147369,0.0


<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The scaled feature matrix retains the same shape as the original windowed dataset, (8121, 65), confirming that the scaling process transforms only the numerical values while preserving the overall structure of the data.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The mean of the scaled data is approximately zero, indicating that the features have been successfully centred. This ensures that the data is balanced around zero, which is beneficial for many machine learning algorithms.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
The standard deviation is approximately 0.88 rather than exactly 1. This is expected in practice due to rounding effects, correlations between features, and the presence of constant or low-variance features (such as minimum values that remain zero). Despite this, the scaling process still effectively normalises the feature space.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, feature scaling ensures that all features contribute more equally during model training, preventing features with larger numeric ranges from dominating the learning process. As a result, the dataset is now well-prepared for downstream machine learning tasks.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">5.4 Final Dataset Summary</h3>
<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
At the end of the preprocessing phase, the original sensor data has been transformed into a structured dataset suitable for machine learning. Through window-based segmentation, the sequential signal was converted into multiple overlapping samples, significantly increasing the number of observations available for analysis.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Each sample consists of a set of engineered features that capture both statistical properties and short-term temporal dynamics of the signal. By combining frame-level features, temporal differences, and rolling statistics within each window, the dataset provides a rich representation of the sensor behaviour.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
In addition, feature scaling was applied to standardise the data, ensuring that all features contribute more equally during model training. This step improves the stability and performance of machine learning algorithms, particularly those sensitive to feature magnitude.
</p>

<p style="font-size:17px; line-height:1.7; max-width:950px; margin:auto; text-align:justify;">
Overall, the preprocessing pipeline transforms the raw sensor measurements into a well-structured, multi-sample, and standardised dataset. This final dataset is now ready to be used for modelling and evaluation in the subsequent phase.
</p>

<h2 style="color:#2874A6; font-weight:bold;">6. Key Insights and Observations</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This feature engineering phase provided several important insights into the behaviour and structure of the CMOS sensor dataset. The transformation from raw 1,024 pixel values per frame into a compact set of engineered features significantly reduced the dimensionality of the data while preserving its most relevant characteristics.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The analysis confirmed that the sensor signal exhibits strong temporal characteristics. Frame-level features showed relatively stable behaviour over short periods, while temporal difference and rolling features revealed that the signal evolves smoothly without abrupt transitions.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The evaluation of potential invalid frames indicated that no additional filtering was required, suggesting that the dataset is already well-structured and contains meaningful signal patterns.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Temporal and rolling features provided complementary perspectives on the signal by capturing both short-term variations and smoothed trends. These features enabled a more detailed understanding of local signal dynamics across consecutive observations.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, a phase-aware segmentation based on heuristic thresholds revealed that the signal can be approximately grouped into different intensity regimes. However, these phase labels should be interpreted with caution, as they do not represent ground truth phases.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the combination of statistical, temporal, rolling, and phase-aware features resulted in a compact and interpretable representation of the sensor data. This feature set provides a strong foundation for subsequent preprocessing and machine learning tasks.
</p>